<a href="https://colab.research.google.com/github/lgelara/prismaFRL/blob/main/Lucia_prisma_paso2_screening_parte2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📋 PRISMA — Paso 2: Screening por Título y Abstract

Este cuaderno aplica los **filtros de elegibilidad** sobre los registros únicos obtenidos en el Paso 1.

### Flujo
```
prisma_unicos.csv
       │
       ▼
  Filtro por TÍTULO  ──── keywords ────►  excluidos_titulo.csv
       │
       ▼
  Filtro automático por ABSTRACT ────►   excluidos_abstract_auto.csv
       │
       ▼
  Revisión MANUAL (abstract vacío
  o score en zona gris) ──────────────►  excluidos_abstract_manual.csv
       │
       ▼
  prisma_elegibles.csv  ✓
```

---
**Instrucciones**
1. Ejecuta la **Celda 1** (dependencias, solo una vez).
2. Edita la **Celda 2** con tus palabras clave y criterios.
3. Ejecuta el resto de celdas en orden.

In [ ]:
# ─────────────────────────────────────────────────────────────
#  CELDA 1 · Dependencias  (ejecutar una sola vez por sesión)
# ─────────────────────────────────────────────────────────────
!pip install pandas openpyxl ipywidgets -q
from google.colab import output
output.enable_custom_widget_manager()
print('✓ Listo')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 17.3 MB/s eta 0:00:00
✓ Listo


In [ ]:
# ─────────────────────────────────────────────────────────────
#  CELDA 2 · CONFIGURACIÓN  ← EDITA AQUÍ
# ─────────────────────────────────────────────────────────────

# ── Palabras clave de inclusión ───────────────────────────────
# Escribe tus términos de búsqueda. No distingue mayúsculas.
# Puedes usar frases completas: 'machine learning', 'deep learning'
KEYWORDS = [
    'Federated Reinforcement Learning',
    'Collaborative Reinforcement Learning',
    'Transformer',
    'OR Attention Mechanism',
    'Overall Equipment Effectiveness',
    'Production Efficency',
    'Stochastic Optimization'
]

# ── Palabras clave de EXCLUSIÓN (opcional) ────────────────────
# Artículos que contengan CUALQUIERA de estos términos serán
# excluidos automáticamente, aunque pasen el filtro de inclusión.
KEYWORDS_EXCLUSION = [
    # 'termino no deseado',
]

# ── Criterio para el filtro de TÍTULO ────────────────────────
#   'any'  → conservar si contiene AL MENOS UNA keyword  ← más permisivo
#   'all'  → conservar si contiene TODAS las keywords    ← más estricto
CRITERIO_TITULO = 'any'

# ── Criterio para el filtro AUTOMÁTICO de ABSTRACT ───────────
#   'any' o 'all'  (mismo significado que arriba)
CRITERIO_ABSTRACT = 'any'

# ── Umbral de score para revisión manual ─────────────────────
# Registros con score de keywords ENTRE 0 y este valor
# (y abstract no vacío) pasarán a revisión manual.
# 0   = solo van a revisión manual los que tienen abstract vacío
# 0.3 = también los que tienen coincidencias bajas (recomendado)
UMBRAL_MANUAL = 0.3

# ── Columnas donde buscar en el CSV ──────────────────────────
# Nombre exacto de la columna de título y abstract en tu CSV.
# Déjalas vacías ('') para detección automática.
COL_TITULO   = 'Document Title'
COL_ABSTRACT = 'Abstract'

# ─────────────────────────────────────────────────────────────
print(f'Keywords de inclusión  : {len(KEYWORDS)} términos')
print(f'Keywords de exclusión  : {len(KEYWORDS_EXCLUSION)} términos')
print(f'Criterio título        : {CRITERIO_TITULO}')
print(f'Criterio abstract      : {CRITERIO_ABSTRACT}')
print(f'Umbral revisión manual : {UMBRAL_MANUAL}')

Keywords de inclusión  : 7 términos
Keywords de exclusión  : 0 términos
Criterio título        : any
Criterio abstract      : any
Umbral revisión manual : 0.3


In [ ]:
# ─────────────────────────────────────────────────────────────
#  CELDA 3 · Cargar keywords desde archivo .txt  (OPCIONAL)
#  Si no tienes archivo, simplemente no ejecutes esta celda.
#  Las keywords del archivo se AGREGAN a las de la Celda 2.
# ─────────────────────────────────────────────────────────────
from google.colab import files

print('Sube un archivo .txt con una keyword por línea:')
subido = files.upload()

for nombre, contenido in subido.items():
    lineas = contenido.decode('utf-8', errors='ignore').splitlines()
    nuevas = [l.strip().lower() for l in lineas if l.strip()]
    antes  = len(KEYWORDS)
    KEYWORDS = list(dict.fromkeys(KEYWORDS + nuevas))  # sin duplicados
    print(f'✓ {nombre}: {len(nuevas)} términos cargados, {len(KEYWORDS) - antes} nuevos agregados')

print(f'\nTotal keywords activas: {len(KEYWORDS)}')
for k in KEYWORDS:
    print(f'   • {k}')

In [ ]:
# ─────────────────────────────────────────────────────────────
#  CELDA 4 · Cargar prisma_unicos.csv
# ─────────────────────────────────────────────────────────────
import pandas as pd
from google.colab import files

print('Sube el archivo prisma_unicos.csv del Paso 1:')
subido = files.upload()

nombre_csv = list(subido.keys())[0]
with open(nombre_csv, 'wb') as f:
    f.write(subido[nombre_csv])

df = pd.read_csv(nombre_csv, encoding='utf-8', on_bad_lines='skip')
print(f'✓ {len(df)} registros cargados, {len(df.columns)} columnas')

# ── Detectar columnas de título y abstract ────────────────────
def detectar_col(df, pistas, override):
    if override and override in df.columns:
        return override
    for p in pistas:
        hits = [c for c in df.columns if p in c.lower()]
        if hits:
            return hits[0]
    return None

col_titulo   = detectar_col(df, ['title','titulo'], COL_TITULO)
col_abstract = detectar_col(df, ['abstract','resumen'], COL_ABSTRACT)

if not col_titulo:
    raise ValueError('No se encontró columna de título. Especifica COL_TITULO en la Celda 2.')

print(f'\nColumna de título   : {col_titulo}')
print(f'Columna de abstract : {col_abstract if col_abstract else "NO ENCONTRADA — solo se filtrará por título"}')
print(f'\nPrimeros títulos:')
for t in df[col_titulo].head(5):
    print(f'  · {str(t)[:90]}')

Sube el archivo prisma_unicos.csv del Paso 1:


Saving prisma_unicos.csv to prisma_unicos.csv
✓ 475 registros cargados, 114 columnas

Columna de título   : Document Title
Columna de abstract : Abstract

Primeros títulos:
  · A Multiagent Federated Reinforcement Learning Approach for Plug-In Electric Vehicle Fleet 
  · A Privacy-Preserving Federated Reinforcement Learning Method for Multiple Virtual Power Pl
  · An Instructional Optimization Method Based on Bidirectional Transformer and Reinforcement 
  · Collaborative Decision-Making Method for Multi-UAV Based on Multiagent Reinforcement Learn
  · Safe and Interpretable Human-Like Planning With Transformer-Based Deep Inverse Reinforceme


In [ ]:
# ─────────────────────────────────────────────────────────────
#  CELDA 5 · Funciones de scoring  (no modificar)
# ─────────────────────────────────────────────────────────────
import re

def limpiar(texto):
    if not isinstance(texto, str):
        return ''
    return texto.lower()

def score_keywords(texto, keywords, criterio='any'):
    """
    Devuelve (pasa: bool, score: float, matches: list)
    score = fracción de keywords encontradas (0.0 – 1.0)
    """
    if not texto or not keywords:
        return False, 0.0, []
    t = limpiar(texto)
    matches = [kw for kw in keywords if re.search(re.escape(kw.lower()), t)]
    score   = len(matches) / len(keywords)
    if criterio == 'any':
        pasa = len(matches) >= 1
    else:  # 'all'
        pasa = len(matches) == len(keywords)
    return pasa, score, matches

def tiene_exclusion(texto, keywords_excl):
    if not keywords_excl or not isinstance(texto, str):
        return False
    t = limpiar(texto)
    return any(re.search(re.escape(kw.lower()), t) for kw in keywords_excl)

print('✓ Funciones de scoring cargadas')

✓ Funciones de scoring cargadas


In [ ]:
# ─────────────────────────────────────────────────────────────
#  CELDA 6 · Filtro por TÍTULO
# ─────────────────────────────────────────────────────────────

resultados_titulo = []
for _, row in df.iterrows():
    titulo = str(row.get(col_titulo, ''))
    excluir_por_neg = tiene_exclusion(titulo, KEYWORDS_EXCLUSION)
    pasa, score, matches = score_keywords(titulo, KEYWORDS, CRITERIO_TITULO)

    if excluir_por_neg:
        decision = 'excluido_titulo_negativo'
    elif pasa:
        decision = 'pasa_titulo'
    else:
        decision = 'excluido_titulo'

    resultados_titulo.append({
        'decision_titulo' : decision,
        'score_titulo'    : round(score, 3),
        'matches_titulo'  : ' | '.join(matches),
    })

df_res = df.copy()
df_res = df_res.assign(**pd.DataFrame(resultados_titulo))

df_pasan_titulo    = df_res[df_res['decision_titulo'] == 'pasa_titulo'].copy()
df_excluidos_titulo = df_res[df_res['decision_titulo'].str.startswith('excluido')].copy()

n_pasan   = len(df_pasan_titulo)
n_excluye = len(df_excluidos_titulo)

print(f'Total registros evaluados : {len(df)}')
print(f'Pasan filtro de título    : {n_pasan}  ({n_pasan/len(df)*100:.1f}%)')
print(f'Excluidos por título      : {n_excluye}  ({n_excluye/len(df)*100:.1f}%)')

if KEYWORDS_EXCLUSION:
    n_neg = len(df_res[df_res['decision_titulo'] == 'excluido_titulo_negativo'])
    print(f'  → por keywords de exclusión : {n_neg}')

print('\nDistribución de scores de título:')
bins = [0, 0.25, 0.5, 0.75, 1.01]
labels = ['0–25%', '26–50%', '51–75%', '76–100%']
df_res['rango_score'] = pd.cut(df_res['score_titulo'], bins=bins, labels=labels, right=False)
display(df_res.groupby('rango_score', observed=True).size().reset_index(name='registros'))

Total registros evaluados : 475
Pasan filtro de título    : 101  (21.3%)
Excluidos por título      : 374  (78.7%)

Distribución de scores de título:


,rango_score,registros
0,0–25%,474
1,26–50%,1


In [ ]:
# ─────────────────────────────────────────────────────────────
#  CELDA 7 · Filtro AUTOMÁTICO por ABSTRACT
# ─────────────────────────────────────────────────────────────

if not col_abstract or col_abstract not in df_pasan_titulo.columns:
    print('⚠ No se encontró columna de abstract.')
    print('  Todos los registros que pasaron el filtro de título van directo a elegibles.')
    df_elegibles_auto   = df_pasan_titulo.copy()
    df_excluidos_abs    = pd.DataFrame()
    df_revision_manual  = pd.DataFrame()
else:
    resultados_abs = []
    for _, row in df_pasan_titulo.iterrows():
        abstract = str(row.get(col_abstract, ''))
        vacio    = abstract.strip() in ('', 'nan', 'None')
        excluir_por_neg = tiene_exclusion(abstract, KEYWORDS_EXCLUSION)
        pasa, score, matches = score_keywords(abstract, KEYWORDS, CRITERIO_ABSTRACT)

        if vacio:
            decision = 'revision_manual'   # sin abstract → revisar
        elif excluir_por_neg:
            decision = 'excluido_abstract_negativo'
        elif pasa and score > UMBRAL_MANUAL:
            decision = 'elegible'
        elif pasa and score <= UMBRAL_MANUAL:
            decision = 'revision_manual'   # score bajo → revisar
        else:
            decision = 'excluido_abstract'

        resultados_abs.append({
            'decision_abstract' : decision,
            'score_abstract'    : round(score, 3),
            'matches_abstract'  : ' | '.join(matches),
            'abstract_vacio'    : vacio,
        })

    df_pasan_titulo = df_pasan_titulo.assign(**pd.DataFrame(resultados_abs, index=df_pasan_titulo.index))

    df_elegibles_auto  = df_pasan_titulo[df_pasan_titulo['decision_abstract'] == 'elegible'].copy()
    df_excluidos_abs   = df_pasan_titulo[df_pasan_titulo['decision_abstract'].str.startswith('excluido')].copy()
    df_revision_manual = df_pasan_titulo[df_pasan_titulo['decision_abstract'] == 'revision_manual'].copy()

    print(f'Registros evaluados en abstract : {len(df_pasan_titulo)}')
    print(f'  ✓ Elegibles automáticos        : {len(df_elegibles_auto)}')
    print(f'  ✗ Excluidos por abstract       : {len(df_excluidos_abs)}')
    print(f'  ? Para revisión manual         : {len(df_revision_manual)}')
    n_vacios = df_revision_manual['abstract_vacio'].sum() if len(df_revision_manual) > 0 else 0
    print(f'    → sin abstract               : {int(n_vacios)}')
    print(f'    → score bajo (≤{UMBRAL_MANUAL})         : {len(df_revision_manual) - int(n_vacios)}')

Registros evaluados en abstract : 101
  ✓ Elegibles automáticos        : 0
  ✗ Excluidos por abstract       : 31
  ? Para revisión manual         : 70
    → sin abstract               : 0
    → score bajo (≤0.3)         : 70


In [ ]:
# ─────────────────────────────────────────────────────────────
#  CELDA 8 · Revisión MANUAL de abstracts en zona gris
#  Usa los botones para decidir cada registro.
#  Al terminar todos, ejecuta la Celda 9.
# ─────────────────────────────────────────────────────────────
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

decisiones_manuales = {}   # idx → 'incluir' | 'excluir'

if len(df_revision_manual) == 0:
    print('✓ No hay registros para revisión manual. Continúa con la Celda 9.')
else:
    indices  = list(df_revision_manual.index)
    total    = len(indices)
    estado   = {'pos': 0}

    # ── Widgets ───────────────────────────────────────────────
    progreso   = widgets.IntProgress(value=0, min=0, max=total,
                                     description='Progreso:',
                                     bar_style='info', style={'bar_color': '#4A90D9'},
                                     layout=widgets.Layout(width='100%'))
    lbl_prog   = widgets.Label(value=f'0 / {total} revisados')
    lbl_titulo = widgets.HTML(value='')
    lbl_score  = widgets.HTML(value='')
    lbl_abs    = widgets.HTML(value='')
    lbl_matches= widgets.HTML(value='')
    btn_inc    = widgets.Button(description='✓  Incluir',
                                button_style='success',
                                layout=widgets.Layout(width='180px', height='40px'))
    btn_exc    = widgets.Button(description='✗  Excluir',
                                button_style='danger',
                                layout=widgets.Layout(width='180px', height='40px'))
    btn_skip   = widgets.Button(description='→  Saltar',
                                button_style='warning',
                                layout=widgets.Layout(width='140px', height='40px'))
    out_msg    = widgets.Output()

    def mostrar_registro(pos):
        if pos >= total:
            lbl_titulo.value = '<b style="color:green">✓ Revisión completada. Ejecuta la Celda 9.</b>'
            lbl_score.value  = ''
            lbl_abs.value    = ''
            lbl_matches.value= ''
            btn_inc.disabled = True
            btn_exc.disabled = True
            btn_skip.disabled= True
            return
        idx = indices[pos]
        row = df_revision_manual.loc[idx]
        titulo   = str(row.get(col_titulo, 'Sin título'))
        abstract = str(row.get(col_abstract, ''))
        score    = row.get('score_abstract', 0)
        matches  = row.get('matches_abstract', '')
        vacio    = row.get('abstract_vacio', False)

        abs_texto = '⚠ Sin abstract disponible' if vacio else (
            abstract[:600] + '...' if len(abstract) > 600 else abstract)

        lbl_titulo.value  = (f'<div style="background:#f0f4ff;border-left:4px solid #4A90D9;'
                             f'padding:10px;margin:8px 0;border-radius:4px">'
                             f'<b>Registro {pos+1} de {total}</b><br>'
                             f'<span style="font-size:15px">{titulo}</span></div>')
        lbl_score.value   = (f'<span style="color:#888">Score de keywords: '
                             f'<b>{score:.0%}</b> &nbsp;|&nbsp; '
                             f'Matches: <b>{matches if matches else "ninguno"}</b></span>')
        lbl_abs.value     = (f'<div style="background:#fafafa;border:1px solid #ddd;'
                             f'padding:10px;border-radius:4px;max-height:220px;'
                             f'overflow-y:auto;font-size:13px;line-height:1.5">'
                             f'{abs_texto}</div>')

    def on_decision(decision):
        def _cb(b):
            pos = estado['pos']
            if pos >= total:
                return
            idx = indices[pos]
            if decision != 'saltar':
                decisiones_manuales[idx] = decision
            estado['pos'] += 1
            progreso.value  = estado['pos']
            lbl_prog.value  = f"{estado['pos']} / {total} revisados  |  incluidos: {sum(1 for v in decisiones_manuales.values() if v=='incluir')}  excluidos: {sum(1 for v in decisiones_manuales.values() if v=='excluir')}  saltados: {estado['pos'] - len(decisiones_manuales)}"
            mostrar_registro(estado['pos'])
        return _cb

    btn_inc.on_click(on_decision('incluir'))
    btn_exc.on_click(on_decision('excluir'))
    btn_skip.on_click(on_decision('saltar'))

    mostrar_registro(0)

    display(widgets.VBox([
        widgets.HBox([progreso, lbl_prog]),
        lbl_titulo,
        lbl_score,
        lbl_abs,
        widgets.HBox([btn_inc, btn_exc, btn_skip]),
        out_msg,
    ]))

    print(f'\nRevisando {total} registros en zona gris.')
    print('Usa → Saltar si quieres decidir después; los saltados quedarán como "pendiente".')


Revisando 70 registros en zona gris.
Usa → Saltar si quieres decidir después; los saltados quedarán como "pendiente".


In [ ]:
# ─────────────────────────────────────────────────────────────
#  CELDA 9 · Consolidar resultados
#  Ejecuta esta celda DESPUÉS de terminar la revisión manual.
# ─────────────────────────────────────────────────────────────

# Aplicar decisiones manuales
if len(df_revision_manual) > 0:
    df_revision_manual = df_revision_manual.copy()
    df_revision_manual['decision_manual'] = df_revision_manual.index.map(
        lambda i: decisiones_manuales.get(i, 'pendiente')
    )
    df_manual_incluidos  = df_revision_manual[df_revision_manual['decision_manual'] == 'incluir']
    df_manual_excluidos  = df_revision_manual[df_revision_manual['decision_manual'] == 'excluir']
    df_manual_pendientes = df_revision_manual[df_revision_manual['decision_manual'] == 'pendiente']
else:
    df_manual_incluidos  = pd.DataFrame()
    df_manual_excluidos  = pd.DataFrame()
    df_manual_pendientes = pd.DataFrame()

# Consolidar elegibles finales
df_elegibles = pd.concat([df_elegibles_auto, df_manual_incluidos], ignore_index=True)
df_excluidos_total = pd.concat([
    df_excluidos_titulo,
    df_excluidos_abs if len(df_excluidos_abs) > 0 else pd.DataFrame(),
    df_manual_excluidos,
], ignore_index=True)

# ── Resumen PRISMA ────────────────────────────────────────────
n_inicio     = len(df)
n_exc_tit    = len(df_excluidos_titulo)
n_pasan_tit  = len(df_pasan_titulo) if 'df_pasan_titulo' in dir() else n_inicio
n_exc_abs    = len(df_excluidos_abs) if 'df_excluidos_abs' in dir() else 0
n_man_inc    = len(df_manual_incluidos)
n_man_exc    = len(df_manual_excluidos)
n_pendientes = len(df_manual_pendientes)
n_elegibles  = len(df_elegibles)

resumen = pd.DataFrame({
    'Etapa PRISMA': [
        'Entrada (post deduplicación)',
        'Excluidos por título',
        'Pasan filtro de título',
        'Excluidos por abstract (auto)',
        'Incluidos en revisión manual',
        'Excluidos en revisión manual',
        'Pendientes de revisión',
        'ELEGIBLES FINALES',
    ],
    'n': [n_inicio, n_exc_tit, n_pasan_tit, n_exc_abs,
          n_man_inc, n_man_exc, n_pendientes, n_elegibles],
})

print('\n══════════════════════════════════════════')
print('  RESUMEN PRISMA — Paso 2: Screening')
print('══════════════════════════════════════════')
display(resumen)

if n_pendientes > 0:
    print(f'\n⚠ Tienes {n_pendientes} registros pendientes de revisión.')
    print('  Vuelve a la Celda 8 y revísalos antes de exportar.')
else:
    print('\n✓ Revisión completa. Ejecuta la Celda 10 para exportar.')


══════════════════════════════════════════
  RESUMEN PRISMA — Paso 2: Screening
══════════════════════════════════════════


,Etapa PRISMA,n
0,Entrada (post deduplicación),475
1,Excluidos por título,374
2,Pasan filtro de título,101
3,Excluidos por abstract (auto),31
4,Incluidos en revisión manual,70
5,Excluidos en revisión manual,0
6,Pendientes de revisión,0
7,ELEGIBLES FINALES,70



✓ Revisión completa. Ejecuta la Celda 10 para exportar.


In [ ]:
# ─────────────────────────────────────────────────────────────
#  CELDA 10 · Exportar resultados
# ─────────────────────────────────────────────────────────────
from google.colab import files

ARCH_ELEGIBLES      = 'prisma_elegibles.csv'
ARCH_EXC_TITULO     = 'prisma_excluidos_titulo.csv'
ARCH_EXC_ABSTRACT   = 'prisma_excluidos_abstract.csv'
ARCH_PENDIENTES     = 'prisma_pendientes_revision.csv'
ARCH_LOG            = 'prisma_log_screening.csv'

df_elegibles.to_csv(ARCH_ELEGIBLES, index=False, encoding='utf-8-sig')
df_excluidos_titulo.to_csv(ARCH_EXC_TITULO, index=False, encoding='utf-8-sig')

if len(df_excluidos_abs) > 0:
    df_excluidos_abs.to_csv(ARCH_EXC_ABSTRACT, index=False, encoding='utf-8-sig')

if n_pendientes > 0:
    df_manual_pendientes.to_csv(ARCH_PENDIENTES, index=False, encoding='utf-8-sig')

resumen.to_csv(ARCH_LOG, index=False, encoding='utf-8-sig')

print('Archivos generados:')
print(f'  ✓  {ARCH_ELEGIBLES}          → {n_elegibles} artículos elegibles')
print(f'  ✓  {ARCH_EXC_TITULO}   → {n_exc_tit} excluidos por título')
if len(df_excluidos_abs) > 0:
    print(f'  ✓  {ARCH_EXC_ABSTRACT}  → {n_exc_abs} excluidos por abstract')
if n_pendientes > 0:
    print(f'  ⚠  {ARCH_PENDIENTES}  → {n_pendientes} pendientes')
print(f'  ✓  {ARCH_LOG}          → log PRISMA')

print('\nDescargando...')
files.download(ARCH_ELEGIBLES)
files.download(ARCH_EXC_TITULO)
if len(df_excluidos_abs) > 0:
    files.download(ARCH_EXC_ABSTRACT)
if n_pendientes > 0:
    files.download(ARCH_PENDIENTES)
files.download(ARCH_LOG)

print('\n✓ Paso 2 completado. Siguiente: evaluación de texto completo.')

Archivos generados:
  ✓  prisma_elegibles.csv          → 70 artículos elegibles
  ✓  prisma_excluidos_titulo.csv   → 374 excluidos por título
  ✓  prisma_excluidos_abstract.csv  → 31 excluidos por abstract
  ✓  prisma_log_screening.csv          → log PRISMA

Descargando...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✓ Paso 2 completado. Siguiente: evaluación de texto completo.
